# Sleep Duration Prediction — Model Selection Competition

## Introduction

This exercise is a Kaggle competition based on a dataset describing sleep quality.

The response variable is `Sleep_Duration`, and the goal is to predict sleep duration from the available descriptors.

**Kaggle username:** Yaxuan Fu

The model selection methods required for this exercise are:
- best subset selection, forward selection, or backward selection
- shrinkage methods

I compare several models using cross-validation and use **mean squared error (MSE)** as the main model selection criterion. After selecting the final model, it is fitted on the full training data and used to predict the hold-out test set for submission to Kaggle.

The final model is selected based on both its cross-validated performance and its predictive performance on the Kaggle hold-out data.

## 1. Data
### 1.1 Load the data

The training data contain the response variable `Sleep_Duration`, while the test data contain the predictors for the hold-out observations.

The `ID` column is treated as an identifier rather than a predictor.

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import KFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn.feature_selection import SequentialFeatureSelector

from sklearn.ensemble import (
    HistGradientBoostingRegressor,
    RandomForestRegressor,
    ExtraTreesRegressor
)

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

print("Training data shape:", train.shape)
print("Test data shape:", test.shape)

display(train.head())

Training data shape: (180, 15)
Test data shape: (194, 14)


,ID,Gender,Age,Occupation,Physical_Activity_Level,Stress_Level,BMI_Category,Heart_Rate,Daily_Steps,Sleep_Disorder,Temp,Mood,History,Quality_of_Sleep,Sleep_Duration
0,311,Female,49,Nurse,90,8,Overweight,75,10000,Sleep Apnea,-1.278341,0.440786,5.407088,7.474116,6.2
1,207,Male,30,Doctor,75,6,Normal,70,8000,NaN,-0.826910,0.691399,5.839617,7.979600,7.7
2,169,Male,35,Engineer,60,4,Normal,65,5000,NaN,1.139555,0.909316,7.039570,5.902686,7.3
3,149,Male,39,Lawyer,60,5,Normal,68,8000,NaN,2.069801,0.741677,4.464130,8.127826,7.2
4,98,Female,50,Nurse,90,8,Overweight,75,10000,Sleep Apnea,-2.907425,0.418233,3.068291,8.099139,6.0


### 1.2 Prepare the predictors

The response variable is `Sleep_Duration`.

The `ID` column is excluded from the predictors because it is an identifier.

Categorical variables are one-hot encoded, while numerical variables are standardized when required by the model.

`handle_unknown="ignore"` allows the preprocessing pipeline to handle categories that may occur in a validation or test set but not in the corresponding training split.

In [3]:
X = train.drop(columns=["Sleep_Duration", "ID"])
y = train["Sleep_Duration"]

categorical_cols = [
    "Gender",
    "Occupation",
    "BMI_Category",
    "Sleep_Disorder"
]

numeric_cols = [
    col for col in X.columns
    if col not in categorical_cols
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numeric_cols
        ),
        (
            "cat",
            OneHotEncoder(
                drop="first",
                handle_unknown="ignore"
            ),
            categorical_cols
        )
    ]
)

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print("Numeric predictors:", numeric_cols)
print("Categorical predictors:", categorical_cols)

Numeric predictors: ['Age', 'Physical_Activity_Level', 'Stress_Level', 'Heart_Rate', 'Daily_Steps', 'Temp', 'Mood', 'History', 'Quality_of_Sleep']
Categorical predictors: ['Gender', 'Occupation', 'BMI_Category', 'Sleep_Disorder']


## 2. Model evaluation

The Kaggle competition evaluates predictions using mean residual sum of squares, i.e. mean squared error (MSE).

Therefore, MSE is used as the main criterion for comparing the models.

Lower MSE indicates better predictive performance.

In [4]:
def cv_mse(model, X_data, y_data):
    scores = -cross_val_score(
        model,
        X_data,
        y_data,
        cv=cv,
        scoring="neg_mean_squared_error"
    )
    return scores.mean(), scores.std()

### 2.1 Linear regression baseline

I first fit an ordinary linear regression model as a baseline.

This model assumes an additive linear relationship between the predictors and sleep duration.

In [5]:
linear_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

linear_mse, linear_sd = cv_mse(
    linear_model,
    X,
    y
)

print(f"Linear Regression CV MSE: {linear_mse:.6f}")
print(f"CV standard deviation: {linear_sd:.6f}")

Linear Regression CV MSE: 0.124837
CV standard deviation: 0.061523


C:\Users\Elena\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


### 2.2 Forward selection

Forward selection starts with no predictors and adds predictors sequentially according to their contribution to predictive performance.

Here the selection is performed using the numerical predictors. The selected variables are then evaluated using 5-fold cross-validation.

In [6]:
X_num = train[numeric_cols]

forward_selector = SequentialFeatureSelector(
    LinearRegression(),
    direction="forward",
    n_features_to_select="auto",
    scoring="neg_mean_squared_error",
    cv=5
)

forward_selector.fit(X_num, y)

forward_features = X_num.columns[
    forward_selector.get_support()
].tolist()

forward_model = LinearRegression()

forward_mse, forward_sd = cv_mse(
    forward_model,
    X_num[forward_features],
    y
)

print("Selected features:")
print(forward_features)
print(f"\nForward Selection CV MSE: {forward_mse:.6f}")
print(f"CV standard deviation: {forward_sd:.6f}")

Selected features:
['Physical_Activity_Level', 'Stress_Level', 'Daily_Steps', 'History']

Forward Selection CV MSE: 0.172809
CV standard deviation: 0.033649


### 2.3 Backward selection

Backward selection starts with all numerical predictors and removes predictors sequentially.

The resulting subset is evaluated using the same 5-fold cross-validation procedure.

In [7]:
backward_selector = SequentialFeatureSelector(
    LinearRegression(),
    direction="backward",
    n_features_to_select="auto",
    scoring="neg_mean_squared_error",
    cv=5
)

backward_selector.fit(X_num, y)

backward_features = X_num.columns[
    backward_selector.get_support()
].tolist()

backward_model = LinearRegression()

backward_mse, backward_sd = cv_mse(
    backward_model,
    X_num[backward_features],
    y
)

print("Selected features:")
print(backward_features)
print(f"\nBackward Selection CV MSE: {backward_mse:.6f}")
print(f"CV standard deviation: {backward_sd:.6f}")

Selected features:
['Physical_Activity_Level', 'Stress_Level', 'Heart_Rate', 'Daily_Steps', 'History']

Backward Selection CV MSE: 0.172205
CV standard deviation: 0.033701


### 2.4 Best subset selection

Best subset selection evaluates different combinations of the numerical predictors.

For each subset size, all possible combinations are considered and evaluated using cross-validation. The subset with the lowest cross-validated MSE is selected.

In [8]:
from itertools import combinations

results = []

for k in range(1, len(numeric_cols) + 1):
    for features in combinations(numeric_cols, k):

        model = LinearRegression()

        mse, _ = cv_mse(
            model,
            X_num[list(features)],
            y
        )

        results.append({
            "n_features": k,
            "features": features,
            "CV_MSE": mse
        })

results_df = pd.DataFrame(results)
results_df = results_df.sort_values("CV_MSE")

display(results_df.head(10))

,n_features,features,CV_MSE
442,6,"(Physical_Activity_Level, Stress_Level, Heart_...",0.170414
495,7,"(Physical_Activity_Level, Stress_Level, Heart_...",0.170571
450,6,"(Physical_Activity_Level, Stress_Level, Daily_...",0.171035
449,6,"(Physical_Activity_Level, Stress_Level, Daily_...",0.171139
340,5,"(Physical_Activity_Level, Stress_Level, Daily_...",0.171167
498,7,"(Physical_Activity_Level, Stress_Level, Daily_...",0.171177
438,6,"(Physical_Activity_Level, Stress_Level, Heart_...",0.172139
327,5,"(Physical_Activity_Level, Stress_Level, Heart_...",0.172205
199,4,"(Physical_Activity_Level, Stress_Level, Histor...",0.172264
470,7,"(Age, Physical_Activity_Level, Stress_Level, H...",0.172389


The best subset contains the following predictors:

`Physical_Activity_Level`, `Stress_Level`, `Heart_Rate`, `Daily_Steps`, `History`, and `Quality_of_Sleep`.

The best subset model has a higher cross-validated MSE than the shrinkage models below, so it is not selected as the final model.

### 2.5 Ridge regression

Ridge regression is a shrinkage method that penalizes large regression coefficients.

The regularization strength is selected by cross-validation.

In [9]:
ridge_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RidgeCV(
        alphas=np.logspace(-4, 4, 100),
        cv=5
    ))
])

ridge_mse, ridge_sd = cv_mse(
    ridge_model,
    X,
    y
)

ridge_model.fit(X, y)

print(f"Ridge CV MSE: {ridge_mse:.6f}")
print(f"CV standard deviation: {ridge_sd:.6f}")
print("Selected alpha:", ridge_model.named_steps["model"].alpha_)

C:\Users\Elena\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


Ridge CV MSE: 0.103551
CV standard deviation: 0.033310
Selected alpha: 1.3219411484660315


### 2.6 Lasso regression

Lasso is another shrinkage method. In addition to shrinking coefficients, it can set some coefficients exactly to zero, providing a form of variable selection.

The regularization strength is selected using cross-validation.

In [10]:
lasso_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LassoCV(
        cv=5,
        max_iter=50000,
        random_state=42
    ))
])

lasso_mse, lasso_sd = cv_mse(
    lasso_model,
    X,
    y
)

lasso_model.fit(X, y)

print(f"Lasso CV MSE: {lasso_mse:.6f}")
print(f"CV standard deviation: {lasso_sd:.6f}")
print("Selected alpha:", lasso_model.named_steps["model"].alpha_)

C:\Users\Elena\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


Lasso CV MSE: 0.096360
CV standard deviation: 0.025627
Selected alpha: 0.003230158449186195


### 2.7 Compare the required model-selection approaches

In [11]:
selection_results = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Forward Selection",
        "Backward Selection",
        "Best Subset",
        "Ridge",
        "Lasso"
    ],
    "CV_MSE": [
        linear_mse,
        forward_mse,
        backward_mse,
        results_df.iloc[0]["CV_MSE"],
        ridge_mse,
        lasso_mse
    ]
})

display(
    selection_results.sort_values("CV_MSE")
)

,Model,CV_MSE
5,Lasso,0.096360
4,Ridge,0.103551
0,Linear Regression,0.124837
3,Best Subset,0.170414
2,Backward Selection,0.172205
1,Forward Selection,0.172809


### Model selection results

Among the linear and shrinkage-based models, Lasso gives the lowest cross-validated MSE.

However, the relatively high MSE compared with the nonlinear models tested below suggests that a purely linear relationship may not be sufficient to describe the data.

## 3. Nonlinear predictive models

After evaluating the required model-selection approaches, I also compare several tree-based models.

These models can capture nonlinear relationships and interactions between predictors that are not represented by ordinary linear regression or shrinkage models.

### 3.1 HistGradientBoosting

In [12]:
hgb_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", HistGradientBoostingRegressor(
        max_iter=300,
        learning_rate=0.05,
        max_leaf_nodes=15,
        l2_regularization=1.0,
        random_state=42
    ))
])

hgb_mse, hgb_sd = cv_mse(
    hgb_model,
    X,
    y
)

print(f"HistGradientBoosting CV MSE: {hgb_mse:.6f}")
print(f"CV standard deviation: {hgb_sd:.6f}")

C:\Users\Elena\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


HistGradientBoosting CV MSE: 0.034627
CV standard deviation: 0.018192


### 3.2 Random Forest

In [13]:
rf_settings = [
    {
        "max_features": "sqrt",
        "min_samples_leaf": 1
    },
    {
        "max_features": "sqrt",
        "min_samples_leaf": 2
    },
    {
        "max_features": "sqrt",
        "min_samples_leaf": 4
    },
    {
        "max_features": 0.5,
        "min_samples_leaf": 2
    },
    {
        "max_features": 1.0,
        "min_samples_leaf": 2
    }
]

rf_results = []

for settings in rf_settings:

    model = Pipeline([
        ("preprocessor", preprocessor),
        ("model", RandomForestRegressor(
            n_estimators=500,
            random_state=42,
            n_jobs=-1,
            **settings
        ))
    ])

    mse, sd = cv_mse(model, X, y)

    rf_results.append({
        **settings,
        "CV_MSE": mse,
        "CV_SD": sd
    })

rf_results_df = pd.DataFrame(rf_results)

display(
    rf_results_df.sort_values("CV_MSE")
)

C:\Users\Elena\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
C:\Users\Elena\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
C:\Users\Elena\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
C:\Users\Elena\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
C:\Users\Elena\anaconda3\Lib\sit

,max_features,min_samples_leaf,CV_MSE,CV_SD
3,0.5,2,0.026446,0.004247
4,1.0,2,0.026750,0.014962
0,sqrt,1,0.029543,0.005217
1,sqrt,2,0.038813,0.008092
2,sqrt,4,0.052382,0.010436


### 3.3 Extra Trees regression

Extra Trees is another tree ensemble method. It introduces additional randomization when constructing the individual trees.

The best-performing configuration from the model comparison uses all available features at each split and a minimum leaf size of 2.

In [14]:
extra_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", ExtraTreesRegressor(
        n_estimators=500,
        max_features=1.0,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    ))
])

extra_mse, extra_sd = cv_mse(
    extra_model,
    X,
    y
)

print(f"Extra Trees CV MSE: {extra_mse:.6f}")
print(f"CV standard deviation: {extra_sd:.6f}")

C:\Users\Elena\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


Extra Trees CV MSE: 0.015682
CV standard deviation: 0.003601


### 3.4 Overall model comparison

In [15]:
all_results = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Forward Selection",
        "Backward Selection",
        "Best Subset",
        "Ridge",
        "Lasso",
        "HistGradientBoosting",
        "Random Forest",
        "Extra Trees"
    ],
    "CV_MSE": [
        linear_mse,
        forward_mse,
        backward_mse,
        results_df.iloc[0]["CV_MSE"],
        ridge_mse,
        lasso_mse,
        hgb_mse,
        rf_results_df["CV_MSE"].min(),
        extra_mse
    ]
})

display(
    all_results.sort_values("CV_MSE")
)

,Model,CV_MSE
8,Extra Trees,0.015682
7,Random Forest,0.026446
6,HistGradientBoosting,0.034627
5,Lasso,0.096360
4,Ridge,0.103551
0,Linear Regression,0.124837
3,Best Subset,0.170414
2,Backward Selection,0.172205
1,Forward Selection,0.172809


The cross-validation results show a clear improvement when moving from linear and shrinkage models to nonlinear tree-based models.

Among the required model-selection approaches, Lasso gives the lowest cross-validated MSE. However, Extra Trees achieves a substantially lower cross-validated MSE than both Lasso and the other models considered.

This suggests that nonlinear relationships and interactions between the descriptors are important for predicting sleep duration.

Therefore, Extra Trees is selected as the final predictive model.

The final model is fitted using the complete training dataset before generating predictions for the Kaggle hold-out data.

## 4. Final Outputs

### 4.1 Fit final model

In [16]:
extra_final = Pipeline([
    ("preprocessor", preprocessor),
    ("model", ExtraTreesRegressor(
        n_estimators=500,
        max_features=1.0,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    ))
])

extra_final.fit(X, y)

final_predictions = extra_final.predict(test)

print("Number of predictions:", len(final_predictions))
print(
    "Prediction range:",
    final_predictions.min(),
    "to",
    final_predictions.max()
)

C:\Users\Elena\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


Number of predictions: 194
Prediction range: 5.878483333333285 to 8.483439047619042


### 4.2 Submission

The final predictions are paired with the corresponding test-set IDs.

The submission file contains exactly two columns: `ID` and `Sleep_Duration`.

In [17]:
submission = pd.DataFrame({
    "ID": test["ID"],
    "Sleep_Duration": final_predictions
})

submission = submission.sort_values("ID")

print("Shape:", submission.shape)

print("\nMissing values:")
print(submission.isna().sum())

print("\nUnique IDs:", submission["ID"].nunique())

print(
    "\nPrediction range:",
    submission["Sleep_Duration"].min(),
    "to",
    submission["Sleep_Duration"].max()
)

display(submission.head(10))

Shape: (194, 2)

Missing values:
ID                0
Sleep_Duration    0
dtype: int64

Unique IDs: 194

Prediction range: 5.878483333333285 to 8.483439047619042


,ID,Sleep_Duration
141,1,6.056576
0,3,6.110520
91,5,6.748543
76,8,6.589778
109,10,7.757310
162,11,7.141366
153,12,6.065245
101,17,6.062747
115,19,8.462350
26,20,7.695942


In [18]:
submission.to_csv(
    "submission_extra_trees1.csv",
    index=False
)

print("Submission file saved as submission_extra_trees1.csv")

Submission file saved as submission_extra_trees1.csv


## Summary

I initially selected Lasso regression as my final model because it was the best-performing shrinkage method among the linear models considered. My first Kaggle submission was therefore based on the Lasso model and achieved a public leaderboard MSE of **0.05511**.

After seeing the Kaggle result, I explored several nonlinear predictive models, including HistGradientBoosting, Random Forest, and Extra Trees. These models performed substantially better in cross-validation, with Extra Trees achieving the lowest cross-validated MSE of approximately **0.0155**.

I therefore selected Extra Trees as the final model and submitted its predictions to Kaggle. The Extra Trees submission achieved a public leaderboard MSE of **0.01596**, which was a substantial improvement over the Lasso submission.

This comparison suggests that nonlinear relationships and interactions between the sleep-related descriptors are important for predicting sleep duration, and that the Extra Trees model was better suited to this dataset than the linear shrinkage model.
